# Imports

In [29]:
import os
import json
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch import amp
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

In [30]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("riondsilva21/hand-keypoint-dataset-26k")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\jonah\.cache\kagglehub\datasets\riondsilva21\hand-keypoint-dataset-26k\versions\3


# DataLoaders

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os


# Credit to Thibaut Juill for the following code, which I adapted for my use case. Original code can be found here: 
# https://www.kaggle.com/code/thibautjuill/cv-hand-key-detection

# Link to Training Data:
# https://www.kaggle.com/datasets/riondsilva21/hand-keypoint-dataset-26k/data


class HandKeypointDataset(Dataset):
    """Dataset pour les keypoints de la main."""
    def __init__(self, image_dir, annotation_file, transform=None, cache=False):
        self.image_dir = image_dir
        self.annotation_file = annotation_file
        self.annotations = self._load_annotations()
        self.transform = transform
        self.cache = cache 
        self.cached_data = {} 

    def _load_annotations(self):
        """Charge les annotations à partir du fichier COCO JSON."""
        with open(self.annotation_file, 'r') as f:
            coco_data = json.load(f)

        self.image_id_to_annotations = {}
        for ann in coco_data['annotations']:
            image_id = ann['image_id']
            if image_id not in self.image_id_to_annotations:
                self.image_id_to_annotations[image_id] = []
            self.image_id_to_annotations[image_id].append(ann)

        image_annotations = []
        for img_data in coco_data['images']:
            image_id = img_data['id']
            image_name = img_data['file_name']
            if image_id in self.image_id_to_annotations:
                annotations = self.image_id_to_annotations[image_id]
                image_annotations.append({'image_id': image_name, 'annotations': annotations})
            else:
                image_annotations.append({'image_id': image_name, 'annotations': []})

        return image_annotations

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        """Retourne l'image et les keypoints à l'index idx."""
        if self.cache and idx in self.cached_data: 
            return self.cached_data[idx] 

        annotation_data = self.annotations[idx]
        image_name = annotation_data['image_id']
        image_path = os.path.join(self.image_dir, image_name)

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  
        image = Image.fromarray(image)
        annotations = annotation_data['annotations']
        keypoints = self._extract_keypoints(annotations)

        if self.transform:
            image = self.transform(image)

        if self.cache:
            self.cached_data[idx] = (image, keypoints) 

        return image, keypoints

    def _extract_keypoints(self, annotations):
        """Extrait les keypoints à partir des annotations COCO."""
        keypoints = torch.zeros((21, 2), dtype=torch.float32)

        for ann in annotations:
            if 'keypoints' in ann and ann['keypoints']:
                kp = ann['keypoints']
                if len(kp) // 3 == 21:
                    for i in range(21):
                        x = kp[i * 3]
                        y = kp[i * 3 + 1]
                        v = kp[i * 3 + 2]
                        if v > 0:
                            keypoints[i, 0] = x
                            keypoints[i, 1] = y
        return keypoints


transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATASET_DIR = 'C:\\Users\\jonah\\.cache\\kagglehub\\datasets\\riondsilva21\\hand-keypoint-dataset-26k\\versions\\3\\hand_keypoint_dataset_26k\\hand_keypoint_dataset_26k'
ANNOTATION_FILE_TRAIN = os.path.join(DATASET_DIR, 'coco_annotation', 'train', '_annotations.coco.json')
ANNOTATION_FILE_VAL = os.path.join(DATASET_DIR, 'coco_annotation', 'val', '_annotations.coco.json')

IMAGE_DIR_TRAIN = os.path.join(DATASET_DIR, 'images', 'train')
IMAGE_DIR_VAL = os.path.join(DATASET_DIR, 'images', 'val')

train_dataset = HandKeypointDataset(IMAGE_DIR_TRAIN, ANNOTATION_FILE_TRAIN, transform=transform_train, cache=True) 
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True) 

val_dataset = HandKeypointDataset(IMAGE_DIR_VAL, ANNOTATION_FILE_VAL, transform=transform_val, cache=True) 
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# Network

In [45]:
class Network(nn.Module):
    def __init__(self):
        super(Network, self).__init__()

        self.conv1 = nn.Conv2d(3, 42, 5)
        self.conv2 = nn.Conv2d(42, 168, 3)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(168, 84)
        self.fc2 = nn.Linear(84, 42)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(self.avgpool(self.conv2_drop(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return F.sigmoid(x).reshape(x.shape[0], 21, 2)





transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


# Train and Test Definition

In [46]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)
model = Network().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

loss_fn = nn.HuberLoss()

def train(epoch):
    model.train()
    
    for batch_idx, (data, target) in enumerate(train_dataloader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data) # prediction
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 1 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_dataset)} ({100. * batch_idx / len(train_dataloader):.0f}%)]\t{loss.item():.6f}')


def test():
    model.eval()

    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in val_dataloader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += loss_fn(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(val_dataset)
    print(f"\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(val_dataset)} ({100. * correct /len(val_dataset):.0f}%)\n")

cpu


In [47]:
train(1)

c:\Users\jonah\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Train Epoch: 1 [0/18776 (0%)]	99.071045
Train Epoch: 1 [32/18776 (0%)]	108.964218
Train Epoch: 1 [64/18776 (0%)]	101.915611
Train Epoch: 1 [96/18776 (1%)]	104.249138
Train Epoch: 1 [128/18776 (1%)]	108.106178
Train Epoch: 1 [160/18776 (1%)]	104.399101
Train Epoch: 1 [192/18776 (1%)]	101.465797
Train Epoch: 1 [224/18776 (1%)]	112.514160
Train Epoch: 1 [256/18776 (1%)]	102.734795
Train Epoch: 1 [288/18776 (2%)]	104.042549
Train Epoch: 1 [320/18776 (2%)]	100.655403
Train Epoch: 1 [352/18776 (2%)]	109.223274
Train Epoch: 1 [384/18776 (2%)]	99.755943
Train Epoch: 1 [416/18776 (2%)]	102.680618
Train Epoch: 1 [448/18776 (2%)]	106.961937
Train Epoch: 1 [480/18776 (3%)]	100.572823
Train Epoch: 1 [512/18776 (3%)]	100.716774
Train Epoch: 1 [544/18776 (3%)]	100.985069
Train Epoch: 1 [576/18776 (3%)]	108.971939
Train Epoch: 1 [608/18776 (3%)]	109.545738
Train Epoch: 1 [640/18776 (3%)]	107.903389
Train Epoch: 1 [672/18776 (4%)]	101.255165
Train Epoch: 1 [704/18776 (4%)]	102.094704
Train Epoch: 1 [73

KeyboardInterrupt: 